In [ ]:
#@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = 'ComfyUI'
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /
    
    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -U --pre comfyui-manager

Download some models/checkpoints/vae or custom comfyui nodes (uncomment the commands for the ones you want)

In [ ]:
# ==========================================
# Download all required models
# ==========================================
import os
import subprocess
from pathlib import Path


def get_hf_token():
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('secretName') or userdata.get('HUGGINGFACE_TOKEN') or userdata.get('hf_token') or userdata.get('huggingface_token')
    except Exception:
        pass
    if not token:
        token = os.environ.get('HUGGINGFACE_TOKEN') or os.environ.get('HF_HUB_TOKEN') or os.environ.get('hf_token') or os.environ.get('huggingface_token')
    return token

hf_token = get_hf_token()
if hf_token:
    os.environ['HUGGINGFACE_TOKEN'] = hf_token
    os.environ['HF_HUB_TOKEN'] = hf_token


def download(url, dest, auth=False):
    Path(dest).parent.mkdir(parents=True, exist_ok=True)
    cmd = ['wget', '-c', url, '-O', dest]
    if auth:
        if hf_token:
            cmd = ['wget', '-c', '--header', f'Authorization: Bearer {hf_token}', url, '-O', dest]
        else:
            print('Warning: auth requested but no Hugging Face token found, trying without auth.')
    print('Downloading', dest)
    subprocess.run(cmd, check=True)

# UNET model (flux1-dev-fp8)
unet_url = 'https://huggingface.co/Kijai/flux-fp8-alternative/resolve/main/flux1-dev-fp8.safetensors'
unet_path = '/content/ComfyUI/models/unet/flux1-dev-fp8.safetensors'
download(unet_url, unet_path, auth=True)

# VAE
vae_url = 'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors'
vae_path = '/content/ComfyUI/models/vae/ae.safetensors'
download(vae_url, vae_path, auth=True)

# Dual CLIP
download('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors', '/content/ComfyUI/models/clip/clip_l.safetensors', auth=False)
download('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors', '/content/ComfyUI/models/clip/t5xxl_fp8_e4m3fn.safetensors', auth=False)

# PuLID Flux
download('https://huggingface.co/guozhipeng/PuLID-Flux/resolve/main/pulid_flux_v0.9.1.safetensors', '/content/ComfyUI/models/pulid/pulid_flux_v0.9.1.safetensors', auth=True)

# EVA-CLIP
download('https://huggingface.co/QuanSun/EVA-CLIP/resolve/main/EVA02_CLIP_L_336_psz14_s6B.pt', '/content/ComfyUI/models/clip_vision/EVA02_CLIP_L_336_psz14_s6B.pt', auth=True)

# InsightFace (deepinsight/antelopev2 - public official repo)
download('https://huggingface.co/deepinsight/antelopev2/resolve/main/1k3d68.onnx', '/root/.insightface/models/antelopev2/1k3d68.onnx', auth=False)
download('https://huggingface.co/deepinsight/antelopev2/resolve/main/2d106det.onnx', '/root/.insightface/models/antelopev2/2d106det.onnx', auth=False)
download('https://huggingface.co/deepinsight/antelopev2/resolve/main/genderage.onnx', '/root/.insightface/models/antelopev2/genderage.onnx', auth=False)
download('https://huggingface.co/deepinsight/antelopev2/resolve/main/glintr100.onnx', '/root/.insightface/models/antelopev2/glintr100.onnx', auth=False)
download('https://huggingface.co/deepinsight/antelopev2/resolve/main/scrfd_10g_bnkps.onnx', '/root/.insightface/models/antelopev2/scrfd_10g_bnkps.onnx', auth=False)

# CLIPSeg
download('https://huggingface.co/CIDAS/clipseg-rd64-refined/resolve/main/pytorch_model.bin', '/content/ComfyUI/models/clipseg/pytorch_model.bin', auth=False)

# Ultralytics face bbox
download('https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt', '/content/ComfyUI/models/ultralytics/bbox/face_yolov8m.pt', auth=False)

print('--- ĐÃ TẢI XONG TOÀN BỘ MODEL CHO WORKFLOW CỦA BẠN! ---')

In [ ]:
# Di chuyển vào thư mục custom_nodes của ComfyUI
%cd /content/ComfyUI/custom_nodes/

# Cài đặt bộ công cụ Impact Pack (Chứa FaceDetailer)
!git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git
%cd ComfyUI-Impact-Pack
!pip install -r requirements.txt
%cd ..

# Cài đặt bộ PuLID Flux Enhanced
!git clone https://github.com/sipie800/ComfyUI-PuLID-Flux-Enhanced.git
%cd ComfyUI-PuLID-Flux-Enhanced
!pip install -r requirements.txt
%cd ..

# Cài đặt WAS Node Suite (Chứa node CLIPSEG2 trong workflow của ông)
!git clone https://github.com/WASasquatch/was-node-suite-comfyui.git

# Quay trở lại thư mục chính của ComfyUI
%cd /content/ComfyUI

### Run ComfyUI with cloudflared (Recommended Way)




In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --enable-manager --dont-print-server

### Run ComfyUI with localtunnel




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --enable-manager --dont-print-server

### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [ ]:
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --enable-manager --dont-print-server